# NB09 — Ablations A2, A4, A5

**Milestone 3 deliverable** — all remaining ablations not covered in NB07/NB08.

| Ablation | Variable | Conditions |
|----------|----------|------------|
| **A2b** | Graph topology (hands-only) | `hands42` — 42-joint hand subgraph |
| **A2c** | Graph topology (late-fusion) | `latefusion` — two 21-joint STGCN branches |
| **A4** | Augmentation | `none` / `spatial` / `temporal` / `full` |
| **A5** | Normalisation | `torso` / `raw` / `bonelength` |

All experiments: **adaptive** topology, **T=32**, **80 epochs**, **patience=10**, **SEED=42**  
Checkpoints → `checkpoints/`  
Results → `results/metrics_all.csv`

In [1]:
import sys, os, time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/Users/yamini/Desktop/projects/ISL PROJECT")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from train import run_training, PROC_DIR, RES_CSV
from graph import _EDGES

HF_TRAIN = PROJECT_ROOT / "include_dataset/data/train-00000-of-00001.parquet"
HF_VAL   = PROJECT_ROOT / "include_dataset/data/val-00000-of-00001.parquet"
HF_TEST  = PROJECT_ROOT / "include_dataset/data/test-00000-of-00001.parquet"

RAW_KP_DIR = PROJECT_ROOT / "data" / "raw_keypoints"

L_SHOULDER, R_SHOULDER = 45, 46
T_TARGET = 64

print("ROOT:", PROJECT_ROOT)
print("PROC_DIR:", PROC_DIR)
print("Parquets exist:", HF_TRAIN.exists(), HF_VAL.exists(), HF_TEST.exists())

ROOT: /Users/yamini/Desktop/projects/ISL PROJECT
PROC_DIR: /Users/yamini/Desktop/projects/ISL PROJECT/data/processed
Parquets exist: True True True


---
## Section 1 — A5 Data Preparation

Build two additional normalisation variants from the raw per-video keypoint files:

- **`X_raw_sd_*.npy`** — gap-fill + resample to T=64, **no** torso centering/scaling  
- **`X_bonelen_sd_*.npy`** — same as raw, then divide by mean inter-joint bone length per sample

Skip if already built.

In [2]:
# ── Helper functions (ported from NB02) ──────────────────────────────────────

def resample_sequence(kps, t_target=T_TARGET):
    T_src = kps.shape[0]
    if T_src == t_target:
        return kps.astype(np.float32)
    src_idx = np.arange(T_src, dtype=np.float32)
    tgt_idx = np.linspace(0, T_src - 1, t_target, dtype=np.float32)
    out = np.zeros((t_target, 53, 3), dtype=np.float32)
    for j in range(53):
        for c in range(3):
            out[:, j, c] = np.interp(tgt_idx, src_idx, kps[:, j, c])
    return out


def fix_missing_hand_v2(kps):
    T = kps.shape[0]
    filled = kps.copy().astype(np.float32)
    mask   = np.zeros((T, 53), dtype=np.float32)
    for j in range(53):
        col      = kps[:, j, :]
        detected = ~np.all(col == 0, axis=1)
        if not detected.any():
            continue
        valid_frames = np.where(detected)[0]
        first, last  = valid_frames[0], valid_frames[-1]
        mask[detected, j] = 1.0
        for c in range(3):
            filled[:, j, c] = np.interp(
                np.arange(T, dtype=np.float64),
                valid_frames.astype(np.float64),
                col[valid_frames, c],
            )
        mask[first : last + 1, j] = 1.0
    return filled, mask


def resample_mask(mask, t_target=T_TARGET):
    T_src = mask.shape[0]
    if T_src == t_target:
        return mask.astype(np.float32)
    src_idx = np.round(np.linspace(0, T_src - 1, t_target)).astype(int)
    return mask[src_idx].astype(np.float32)


def load_raw(npy_name):
    """Gap-fill + resample; NO torso normalization."""
    path = RAW_KP_DIR / npy_name
    if not path.exists():
        return None, None
    kps = np.load(str(path))
    if kps.shape[0] < 2:
        return None, None
    kps, mask = fix_missing_hand_v2(kps)
    kps  = resample_sequence(kps, T_TARGET)
    mask = resample_mask(mask, T_TARGET)
    return kps, mask


print("Helper functions defined.")

Helper functions defined.


In [3]:
# ── Check if raw/bonelength files already exist ───────────────────────────────

_raw_ok = all(
    (PROC_DIR / f"X_raw_sd_{s}.npy").exists()
    for s in ("train", "val", "test")
)
_bl_ok = all(
    (PROC_DIR / f"X_bonelen_sd_{s}.npy").exists()
    for s in ("train", "val", "test")
)

print(f"X_raw files exist:     {_raw_ok}")
print(f"X_bonelen files exist: {_bl_ok}")

if _raw_ok and _bl_ok:
    print("Both sets already built — skipping data preparation.")
else:
    print("Building missing normalization variants...")

X_raw files exist:     True
X_bonelen files exist: True
Both sets already built — skipping data preparation.


In [4]:
if not _raw_ok:
    import pickle
    from sklearn.preprocessing import LabelEncoder

    # ── Load parquet + build path_to_npy mapping (same as NB02) ─────────────
    meta_df = pd.read_csv(PROJECT_ROOT / "data" / "keypoints_metadata.csv")
    meta_df["rel_path"] = (
        meta_df["category"] + "/" + meta_df["sign"] + "/" + meta_df["video_id"] + ".MOV"
    )
    path_to_npy = dict(zip(meta_df["rel_path"], meta_df["out_name"]))
    print(f"path_to_npy: {len(path_to_npy)} entries")

    with open(PROC_DIR / "label_encoder.pkl", "rb") as f:
        le = pickle.load(f)

    # ── Cross-split dedup (identical to NB02) ────────────────────────────────
    df_train_raw = pd.read_parquet(str(HF_TRAIN))
    df_val_raw   = pd.read_parquet(str(HF_VAL))
    df_test_raw  = pd.read_parquet(str(HF_TEST))

    test_vp = set(df_test_raw["video_path"])
    val_vp  = set(df_val_raw["video_path"])

    df_test  = df_test_raw
    df_val   = df_val_raw[~df_val_raw["video_path"].isin(test_vp)].reset_index(drop=True)
    df_train = df_train_raw[
        ~df_train_raw["video_path"].isin(test_vp | val_vp)
    ].reset_index(drop=True)
    print(f"Split sizes: train={len(df_train)}, val={len(df_val)}, test={len(df_test)}")

    def build_raw_split(df, split_name):
        X_list, y_list = [], []
        seen_npy = set()
        skipped  = 0
        for i, (_, row) in enumerate(df.iterrows()):
            if i % 500 == 0:
                print(f"  [{split_name}] {i}/{len(df)}", end="\r")
            npy_name = path_to_npy.get(row["video_path"])
            if npy_name is None or npy_name in seen_npy:
                skipped += 1
                continue
            seen_npy.add(npy_name)
            kps, _ = load_raw(npy_name)
            if kps is None:
                skipped += 1
                continue
            try:
                label_int = le.transform([row["label"]])[0]
            except ValueError:
                skipped += 1
                continue
            X_list.append(kps)
            y_list.append(label_int)
        print(f"  [{split_name}] done: {len(X_list)} samples, {skipped} skipped")
        return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.int64)

    t0 = time.time()
    X_raw_tr, y_tr = build_raw_split(df_train, "train")
    X_raw_va, y_va = build_raw_split(df_val,   "val")
    X_raw_te, y_te = build_raw_split(df_test,  "test")
    print(f"\nBuilt X_raw splits in {(time.time()-t0)/60:.1f} min")
    print(f"  train: {X_raw_tr.shape}, val: {X_raw_va.shape}, test: {X_raw_te.shape}")

    # Verify label ordering matches existing y_sd_*.npy
    y_sd_tr = np.load(PROC_DIR / "y_sd_train.npy")
    assert np.array_equal(y_tr, y_sd_tr), "Label mismatch between raw and torso splits!"
    print("Label ordering verified — matches y_sd_train.npy")

    np.save(PROC_DIR / "X_raw_sd_train.npy", X_raw_tr)
    np.save(PROC_DIR / "X_raw_sd_val.npy",   X_raw_va)
    np.save(PROC_DIR / "X_raw_sd_test.npy",  X_raw_te)
    print("Saved X_raw_sd_*.npy")
    _raw_ok = True

In [5]:
if not _bl_ok:
    # ── Build bone-length normalised arrays ────────────────────────────────
    # Mean inter-joint distance is computed per sample across all frames
    # and all bone edges (60 edges in the 53-joint graph).
    # Each sample is divided by its own mean bone length.

    edges = np.array(_EDGES, dtype=np.int64)   # (60, 2)

    def bonelength_normalize(X):
        """X: (N, T, V, 3) → (N, T, V, 3) scaled per sample."""
        # Bone vectors: (N, T, E, 3)
        bone_vecs = X[:, :, edges[:, 0], :] - X[:, :, edges[:, 1], :]
        # Bone lengths: (N, T, E)
        bone_lens = np.linalg.norm(bone_vecs, axis=-1)
        # Per-sample mean bone length: (N,)
        mean_bl = bone_lens.mean(axis=(1, 2))   # mean over T and E
        # Avoid division by zero
        mean_bl = np.where(mean_bl < 1e-6, 1.0, mean_bl)
        # Scale: (N, 1, 1, 1)
        return (X / mean_bl[:, None, None, None]).astype(np.float32)

    print("Computing bone-length normalised variants...")
    for split in ("train", "val", "test"):
        raw_path = PROC_DIR / f"X_raw_sd_{split}.npy"
        if not raw_path.exists():
            print(f"  ERROR: {raw_path} not found — build raw first.")
            break
        X_raw = np.load(str(raw_path))
        X_bl  = bonelength_normalize(X_raw)
        np.save(PROC_DIR / f"X_bonelen_sd_{split}.npy", X_bl)
        print(f"  {split}: {X_bl.shape}  mean_bl example={np.linalg.norm(X_raw[:, :, edges[:,0], :] - X_raw[:, :, edges[:,1], :], axis=-1).mean(axis=(1,2))[:3].round(4)}")
    _bl_ok = True
    print("Saved X_bonelen_sd_*.npy")

In [6]:
# ── Sanity check: compare value ranges across normalisations ─────────────────
print("Value ranges for test split (sample 0, joint 0):")
for norm, prefix in [("torso", "X_sd"), ("raw", "X_raw_sd"), ("bonelength", "X_bonelen_sd")]:
    X = np.load(PROC_DIR / f"{prefix}_test.npy")
    print(f"  {norm:12s}: shape={X.shape}  min={X.min():.3f}  max={X.max():.3f}  "
          f"mean={X.mean():.3f}  std={X.std():.3f}")

Value ranges for test split (sample 0, joint 0):
  torso       : shape=(858, 64, 53, 3)  min=-5.848  max=5.031  mean=0.824  std=1.424
  raw         : shape=(858, 64, 53, 3)  min=-1.078  max=1.061  mean=0.354  std=0.332
  bonelength  : shape=(858, 64, 53, 3)  min=-13.138  max=15.608  mean=5.015  std=4.711


---
## Section 2 — A2b: Hands-Only (42-joint) Topology

**Motivation:** Upper-body joints 42-52 are less discriminative for sign language (mostly static shoulder/elbow positions). This ablation tests whether restricting the graph to the two hand subgraphs (21 joints each, 42 total) improves recognition.

**Config:** `topology=hands42`, `T=32`, `augmentation=full`, `normalisation=torso`, 80 epochs, patience=10  
**Model:** STGCN with 42-joint spatial adjacency (L_wrist→R_wrist inter-hand bridge replacing shoulder chain)

In [7]:
print("=" * 60)
print("A2b: hands42 topology — T=32, 80ep, full aug, torso norm")
print("=" * 60)

result_hands42 = run_training(
    topology      = "hands42",
    T             = 32,
    augmentation  = "full",
    normalisation = "torso",
    epochs        = 80,
    patience      = 10,
    verbose       = True,
)
print(f"\nA2b result: {result_hands42}")

A2b: hands42 topology — T=32, 80ep, full aug, torso norm

  hands42_T32_full_torso  |  params=2,391,246  |  device=mps
  ep   1/80  tr=0.0032  vl=0.0181  (0.2m)
  ep   2/80  tr=0.0211  vl=0.0301  (0.3m)
  ep   3/80  tr=0.0382  vl=0.0723  (0.5m)
  ep   4/80  tr=0.0577  vl=0.1145  (0.6m)
  ep   5/80  tr=0.0780  vl=0.1084  (0.8m)
  ep   6/80  tr=0.0881  vl=0.1476  (0.9m)
  ep   7/80  tr=0.1170  vl=0.1747  (1.1m)
  ep   8/80  tr=0.1348  vl=0.2139  (1.2m)
  ep   9/80  tr=0.1641  vl=0.2078  (1.4m)
  ep  10/80  tr=0.1881  vl=0.2289  (1.5m)
  ep  11/80  tr=0.2262  vl=0.2892  (1.7m)
  ep  12/80  tr=0.2372  vl=0.2892  (1.8m)
  ep  13/80  tr=0.2762  vl=0.3193  (2.0m)
  ep  14/80  tr=0.2920  vl=0.3283  (2.1m)
  ep  15/80  tr=0.3148  vl=0.3735  (2.3m)
  ep  16/80  tr=0.3546  vl=0.3855  (2.4m)
  ep  17/80  tr=0.3542  vl=0.4699  (2.6m)
  ep  18/80  tr=0.3968  vl=0.3825  (2.7m)
  ep  19/80  tr=0.4212  vl=0.4639  (2.9m)
  ep  20/80  tr=0.4464  vl=0.4458  (3.0m)
  ep  21/80  tr=0.4444  vl=0.4458  (3.2m)

---
## Section 3 — A2c: Late-Fusion (Two 21-Joint Branches)

**Motivation:** A shared 53-joint graph treats both hands symmetrically. Two independent 21-joint STGCN branches (one per hand) can learn hand-specific patterns before fusing via concatenation.

**Config:** `topology=latefusion`, `T=32`, `augmentation=full`, `normalisation=torso`, 80 epochs, patience=10  
**Model:** `LateFusionSTGCN` — left branch (joints 0-20) + right branch (joints 21-41) → concat 512-d → FC(512, 262)

In [8]:
print("=" * 60)
print("A2c: late-fusion topology — T=32, 80ep, full aug, torso norm")
print("=" * 60)

result_latefusion = run_training(
    topology      = "latefusion",
    T             = 32,
    augmentation  = "full",
    normalisation = "torso",
    epochs        = 80,
    patience      = 10,
    verbose       = True,
)
print(f"\nA2c result: {result_latefusion}")

A2c: late-fusion topology — T=32, 80ep, full aug, torso norm

  latefusion_T32_full_torso  |  params=4,710,536  |  device=mps
  ep   1/80  tr=0.0032  vl=0.0120  (0.2m)
  ep   2/80  tr=0.0175  vl=0.0241  (0.3m)
  ep   3/80  tr=0.0264  vl=0.0422  (0.5m)
  ep   4/80  tr=0.0406  vl=0.0392  (0.7m)
  ep   5/80  tr=0.0556  vl=0.0301  (0.8m)
  ep   6/80  tr=0.0617  vl=0.0633  (1.0m)
  ep   7/80  tr=0.0885  vl=0.1235  (1.1m)
  ep   8/80  tr=0.1129  vl=0.1295  (1.3m)
  ep   9/80  tr=0.1320  vl=0.0843  (1.5m)
  ep  10/80  tr=0.1511  vl=0.1416  (1.6m)
  ep  11/80  tr=0.1657  vl=0.2319  (1.8m)
  ep  12/80  tr=0.1970  vl=0.2199  (1.9m)
  ep  13/80  tr=0.2283  vl=0.2620  (2.1m)
  ep  14/80  tr=0.2579  vl=0.2470  (2.3m)
  ep  15/80  tr=0.2762  vl=0.2982  (2.4m)
  ep  16/80  tr=0.3115  vl=0.2922  (2.6m)
  ep  17/80  tr=0.3278  vl=0.3283  (2.8m)
  ep  18/80  tr=0.3627  vl=0.4217  (2.9m)
  ep  19/80  tr=0.3802  vl=0.4096  (3.1m)
  ep  20/80  tr=0.3781  vl=0.3886  (3.3m)
  ep  21/80  tr=0.4192  vl=0.4277 

In [9]:
# ── A2 topology ablation full summary ─────────────────────────────────────────
df_all = pd.read_csv(RES_CSV)

a2_prior = [
    {"condition": "single (T=32)",   "val_acc": 0.7380, "test_acc": 0.7413, "macro_f1": 0.7388},
    {"condition": "dual (T=32)",     "val_acc": 0.7199, "test_acc": 0.7284, "macro_f1": 0.7129},
    {"condition": "adaptive (T=32)", "val_acc": 0.7651, "test_acc": 0.7459, "macro_f1": 0.7517},
]
a2_new = [
    {"condition": "hands42 (T=32)",    **{k: result_hands42[k]   for k in ("val_acc","test_acc","macro_f1")}},
    {"condition": "latefusion (T=32)", **{k: result_latefusion[k] for k in ("val_acc","test_acc","macro_f1")}},
]
a2_df = pd.DataFrame(a2_prior + a2_new)
print("\n=== A2 — Graph Topology Ablation (equal budget: T=32, 80ep, patience=10) ===")
print(a2_df.to_string(index=False, float_format="{:.4f}".format))


=== A2 — Graph Topology Ablation (equal budget: T=32, 80ep, patience=10) ===
        condition  val_acc  test_acc  macro_f1
    single (T=32)   0.7380    0.7413    0.7388
      dual (T=32)   0.7199    0.7284    0.7129
  adaptive (T=32)   0.7651    0.7459    0.7517
   hands42 (T=32)   0.7199    0.7436    0.7440
latefusion (T=32)   0.7560    0.7389    0.7304


---
## Section 4 — A4: Augmentation Ablation

**Motivation:** Identify which augmentation strategy contributes most to generalisation.

| Condition | Joint noise (σ=0.01) | Temporal crop (10%) | H-flip (p=0.5) | Rotation (±15°) |
|-----------|---------------------|---------------------|----------------|----------------|
| `none`     | ✗ | ✗ | ✗ | ✗ |
| `spatial`  | ✓ | ✗ | ✗ | ✗ |
| `temporal` | ✗ | ✓ | ✗ | ✗ |
| `full`     | ✓ | ✓ | ✓ | ✓ |

All at: **adaptive**, T=32, torso norm, 80 epochs, patience=10

> **Note:** Horizontal flip (`_random_horizontal_flip`) and rotation (`_random_rotation`, ±15°) were
> added to `dataset.py` during Milestone 3. Re-running this section produces definitive A4 numbers.
> The `none` condition is unaffected (augmentation off).

In [10]:
a4_results = {}

for aug in ("none", "spatial", "temporal", "full"):
    print(f"\n{'='*60}")
    print(f"A4: augmentation={aug}")
    print(f"{'='*60}")
    result = run_training(
        topology      = "adaptive",
        T             = 32,
        augmentation  = aug,
        normalisation = "torso",
        epochs        = 80,
        patience      = 10,
        verbose       = True,
    )
    a4_results[aug] = result
    print(f"  Done: val={result['val_acc']:.4f}  test={result['test_acc']:.4f}  F1={result['macro_f1']:.4f}")

print("\nAll A4 conditions trained.")


A4: augmentation=none

  adaptive_T32_none_torso  |  params=2,495,370  |  device=mps
  ep   1/80  tr=0.0061  vl=0.0181  (0.2m)
  ep   2/80  tr=0.0142  vl=0.0361  (0.4m)
  ep   3/80  tr=0.0288  vl=0.0482  (0.6m)
  ep   4/80  tr=0.0589  vl=0.0753  (0.8m)
  ep   5/80  tr=0.0975  vl=0.1566  (1.0m)
  ep   6/80  tr=0.1353  vl=0.2169  (1.2m)
  ep   7/80  tr=0.1942  vl=0.1928  (1.4m)
  ep   8/80  tr=0.2441  vl=0.2048  (1.6m)
  ep   9/80  tr=0.2900  vl=0.3584  (1.8m)
  ep  10/80  tr=0.3241  vl=0.3645  (2.1m)
  ep  11/80  tr=0.3859  vl=0.4096  (2.3m)
  ep  12/80  tr=0.4265  vl=0.4789  (2.5m)
  ep  13/80  tr=0.4838  vl=0.4940  (2.7m)
  ep  14/80  tr=0.5183  vl=0.5422  (2.9m)
  ep  15/80  tr=0.5646  vl=0.5422  (3.1m)
  ep  16/80  tr=0.6149  vl=0.6024  (3.3m)
  ep  17/80  tr=0.6369  vl=0.6114  (3.5m)
  ep  18/80  tr=0.6613  vl=0.6747  (3.7m)
  ep  19/80  tr=0.6864  vl=0.6837  (3.9m)
  ep  20/80  tr=0.7307  vl=0.6988  (4.1m)
  ep  21/80  tr=0.7405  vl=0.6295  (4.3m)
  ep  22/80  tr=0.7693  vl=0.768

In [11]:
# ── A4 results table ──────────────────────────────────────────────────────────
a4_rows = []
aug_descriptions = {
    "none":     "No augmentation",
    "spatial":  "Joint noise only (σ=0.01)",
    "temporal": "Temporal crop only (10%)",
    "full":     "Joint noise + temporal crop + horizontal flip (p=0.5) + rotation (±15°)",
}
for aug, res in a4_results.items():
    a4_rows.append({
        "condition":   aug,
        "description": aug_descriptions[aug],
        "val_acc":     res["val_acc"],
        "test_acc":    res["test_acc"],
        "macro_f1":    res["macro_f1"],
    })

a4_df = pd.DataFrame(a4_rows)
a4_df = a4_df.sort_values("test_acc", ascending=False)
print("\n=== A4 — Augmentation Ablation (adaptive, T=32, torso) ===")
print(a4_df.to_string(index=False, float_format="{:.4f}".format))

# Save standalone CSV
a4_df.to_csv(PROJECT_ROOT / "results" / "ablation_A4_augmentation.csv", index=False)
print("\nSaved results/ablation_A4_augmentation.csv")


=== A4 — Augmentation Ablation (adaptive, T=32, torso) ===
condition                                           description  val_acc  test_acc  macro_f1
  spatial                             Joint noise only (σ=0.01)   0.8946    0.9033    0.9012
     none                                       No augmentation   0.9036    0.8986    0.8928
     full Joint noise + temporal crop + horizontal flip (p=0.5)   0.7500    0.7378    0.7323
 temporal                              Temporal crop only (10%)   0.7530    0.7319    0.7266

Saved results/ablation_A4_augmentation.csv


---
## Section 5 — A5: Normalisation Ablation

**Motivation:** Torso normalisation (subtract shoulder midpoint, divide by shoulder width) is standard for body-pose recognition (Yan et al. 2018). For hand-dominant sign language, it may introduce more variance (shoulders often static) or distort hand scales. We test three variants:

| Condition | Description | Status |
|-----------|-------------|--------|
| `torso` | Per-frame: subtract shoulder midpoint / shoulder width | Reused from `temporal,adaptive-T32` |
| `raw` | Raw MediaPipe coordinates (gap-fill + resample only) | **New run** |
| `bonelength` | Divide raw by per-sample mean inter-joint bone length | **New run** |

All at: **adaptive**, T=32, full augmentation, 80 epochs, patience=10

In [12]:
a5_results = {}

# torso already trained: temporal,adaptive-T32 (val=0.7651, test=0.7459, F1=0.7517)
df_csv = pd.read_csv(RES_CSV)
_torso_row = df_csv[(df_csv["experiment"] == "temporal") & (df_csv["condition"] == "adaptive-T32")]
a5_results["torso"] = {
    "val_acc":  float(_torso_row["val_acc"].iloc[0]),
    "test_acc": float(_torso_row["test_acc"].iloc[0]),
    "macro_f1": float(_torso_row["macro_f1"].iloc[0]),
}
print(f"torso — reused from CSV: {a5_results['torso']}")

for norm in ("raw", "bonelength"):
    print(f"\n{'='*60}")
    print(f"A5: normalisation={norm}")
    print(f"{'='*60}")

    from train import _NORM_PREFIX
    npy_path = PROC_DIR / f"{_NORM_PREFIX[norm]}_train.npy"
    if not npy_path.exists():
        print(f"  SKIP: {npy_path} not found — run Section 1 first.")
        continue

    result = run_training(
        topology      = "adaptive",
        T             = 32,
        augmentation  = "full",
        normalisation = norm,
        epochs        = 80,
        patience      = 10,
        verbose       = True,
    )
    a5_results[norm] = result
    print(f"  Done: val={result['val_acc']:.4f}  test={result['test_acc']:.4f}  F1={result['macro_f1']:.4f}")

print("\nAll A5 conditions done.")

torso — reused from CSV: {'val_acc': 0.7651, 'test_acc': 0.7716, 'macro_f1': 0.769}

A5: normalisation=raw

  adaptive_T32_full_raw  |  params=2,495,370  |  device=mps
  ep   1/80  tr=0.0037  vl=0.0060  (0.2m)
  ep   2/80  tr=0.0097  vl=0.0241  (0.4m)
  ep   3/80  tr=0.0158  vl=0.0151  (0.6m)
  ep   4/80  tr=0.0203  vl=0.0301  (0.8m)
  ep   5/80  tr=0.0256  vl=0.0361  (1.0m)
  ep   6/80  tr=0.0353  vl=0.0301  (1.2m)
  ep   7/80  tr=0.0435  vl=0.0633  (1.4m)
  ep   8/80  tr=0.0544  vl=0.0633  (1.6m)
  ep   9/80  tr=0.0609  vl=0.0904  (1.8m)
  ep  10/80  tr=0.0788  vl=0.1024  (2.0m)
  ep  11/80  tr=0.0971  vl=0.1416  (2.3m)
  ep  12/80  tr=0.1040  vl=0.1175  (2.5m)
  ep  13/80  tr=0.1300  vl=0.1596  (2.7m)
  ep  14/80  tr=0.1466  vl=0.1928  (2.9m)
  ep  15/80  tr=0.1580  vl=0.1777  (3.1m)
  ep  16/80  tr=0.1824  vl=0.1687  (3.3m)
  ep  17/80  tr=0.1889  vl=0.1898  (3.5m)
  ep  18/80  tr=0.2108  vl=0.2289  (3.7m)
  ep  19/80  tr=0.2266  vl=0.2289  (3.9m)
  ep  20/80  tr=0.2628  vl=0.2349 

In [13]:
# ── A5 results table ──────────────────────────────────────────────────────────
norm_descriptions = {
    "torso":      "Subtract shoulder midpoint / shoulder width",
    "raw":        "Raw MediaPipe coordinates (no normalisation)",
    "bonelength": "Divide by mean inter-joint bone length per sample",
}
a5_rows = []
for norm, res in a5_results.items():
    a5_rows.append({
        "condition":   norm,
        "description": norm_descriptions.get(norm, norm),
        "val_acc":     res["val_acc"],
        "test_acc":    res["test_acc"],
        "macro_f1":    res["macro_f1"],
    })

a5_df = pd.DataFrame(a5_rows)
a5_df = a5_df.sort_values("test_acc", ascending=False)
print("\n=== A5 — Normalisation Ablation (adaptive, T=32, full aug) ===")
print(a5_df.to_string(index=False, float_format="{:.4f}".format))

a5_df.to_csv(PROJECT_ROOT / "results" / "ablation_A5_normalisation.csv", index=False)
print("\nSaved results/ablation_A5_normalisation.csv")


=== A5 — Normalisation Ablation (adaptive, T=32, full aug) ===
 condition                                       description  val_acc  test_acc  macro_f1
     torso       Subtract shoulder midpoint / shoulder width   0.7651    0.7716    0.7690
       raw      Raw MediaPipe coordinates (no normalisation)   0.5934    0.6294    0.6228
bonelength Divide by mean inter-joint bone length per sample   0.5873    0.6154    0.6088

Saved results/ablation_A5_normalisation.csv


---
## Section 6 — Full Results Summary

All ablation results compiled from `results/metrics_all.csv` plus individual ablation CSVs.

In [14]:
# ── Full metrics_all.csv ──────────────────────────────────────────────────────
df_all = pd.read_csv(RES_CSV)
print("Current metrics_all.csv:")
print(df_all.to_string(index=False, float_format="{:.4f}".format))

Current metrics_all.csv:
experiment                condition  T  val_acc  test_acc  macro_f1
  baseline                   BiLSTM 64      NaN    0.5660    0.5510
  baseline                   1D-CNN 64      NaN    0.8650    0.8650
  topology             single-graph 64   0.5452    0.5361    0.5120
  topology               dual-graph 64   0.3645    0.3660    0.3230
  topology                 adaptive 64   0.5783    0.5886    0.5683
  temporal             adaptive-T32 32   0.7651    0.7716    0.7690
  temporal             adaptive-T48 48   0.6145    0.6026    0.5889
  temporal             adaptive-T64 64   0.5753    0.5828    0.5664
  temporal             adaptive-T96 96   0.0181    0.0070    0.0001
   leakage             1D-CNN-clean 64      NaN    0.8650       NaN
   leakage            1D-CNN-leaked 64      NaN    0.9324    0.9011
   leakage   1D-CNN-leaked-on-clean 64      NaN    0.9312    0.9014
  topology             single-graph 32   0.0271    0.0186    0.0030
  topology             

In [15]:
# ── Ablation tables per research question ────────────────────────────────────

print("\n" + "="*65)
print("RQ1 — Does the HF parquet leakage inflate benchmark numbers?")
print("="*65)
leakage_rows = df_all[df_all["experiment"] == "leakage"][["condition","test_acc","macro_f1"]]
print(leakage_rows.to_string(index=False, float_format="{:.4f}".format))

print("\n" + "="*65)
print("RQ2 — Temporal window (A1)")
print("="*65)
a1_rows = df_all[df_all["experiment"] == "temporal"][["condition","T","val_acc","test_acc","macro_f1"]]
print(a1_rows.to_string(index=False, float_format="{:.4f}".format))

print("\n" + "="*65)
print("RQ3 — Graph topology (A2: single/dual/adaptive/hands42/latefusion)")
print("="*65)
a2_rows_all = df_all[df_all["experiment"].isin(["topology", "ablation"])][
    ["condition","T","val_acc","test_acc","macro_f1"]
]
print(a2_rows_all.to_string(index=False, float_format="{:.4f}".format))

print("\n" + "="*65)
print("A4 — Augmentation")
print("="*65)
try:
    a4_df_out = pd.read_csv(PROJECT_ROOT / "results" / "ablation_A4_augmentation.csv")
    print(a4_df_out.to_string(index=False, float_format="{:.4f}".format))
except FileNotFoundError:
    print("ablation_A4_augmentation.csv not found — run Section 4.")

print("\n" + "="*65)
print("A5 — Normalisation")
print("="*65)
try:
    a5_df_out = pd.read_csv(PROJECT_ROOT / "results" / "ablation_A5_normalisation.csv")
    print(a5_df_out.to_string(index=False, float_format="{:.4f}".format))
except FileNotFoundError:
    print("ablation_A5_normalisation.csv not found — run Section 5.")


RQ1 — Does the HF parquet leakage inflate benchmark numbers?
             condition  test_acc  macro_f1
          1D-CNN-clean    0.8650       NaN
         1D-CNN-leaked    0.9324    0.9011
1D-CNN-leaked-on-clean    0.9312    0.9014

RQ2 — Temporal window (A1)
   condition  T  val_acc  test_acc  macro_f1
adaptive-T32 32   0.7651    0.7716    0.7690
adaptive-T48 48   0.6145    0.6026    0.5889
adaptive-T64 64   0.5753    0.5828    0.5664
adaptive-T96 96   0.0181    0.0070    0.0001

RQ3 — Graph topology (A2: single/dual/adaptive/hands42/latefusion)
               condition  T  val_acc  test_acc  macro_f1
            single-graph 64   0.5452    0.5361    0.5120
              dual-graph 64   0.3645    0.3660    0.3230
                adaptive 64   0.5783    0.5886    0.5683
            single-graph 32   0.0271    0.0186    0.0030
              dual-graph 32   0.7319    0.7448    0.7437
      hands42_full_torso 32   0.7199    0.7436    0.7440
   latefusion_full_torso 32   0.7560    0.7389

In [16]:
# ── Acceptance check: ST-GCN beats LSTM on ≥1 protocol ───────────────────────
stgcn_best_test = df_all[df_all["experiment"].isin(["temporal","topology"])][
    "test_acc"
].max()
bilstm_test  = 0.566
cnn1d_test   = 0.865

print("\n=== Milestone 3 acceptance check ===")
print(f"  ST-GCN best test acc : {stgcn_best_test:.4f}")
print(f"  BiLSTM test acc      : {bilstm_test:.4f}")
print(f"  1D-CNN test acc      : {cnn1d_test:.4f}")
print(f"  ST-GCN > BiLSTM?     : {'YES ✓' if stgcn_best_test > bilstm_test else 'NO'}")
print(f"  ST-GCN > 1D-CNN?     : {'YES ✓' if stgcn_best_test > cnn1d_test  else 'NO  (ok — see thesis rationale)'}")
print()
print("  Roadmap requirement: 'ST-GCN beats LSTM on ≥1 protocol' — ",
      "SATISFIED" if stgcn_best_test > bilstm_test else "NOT MET")
print()
print("Checkpoints in:", sorted([p.name for p in (PROJECT_ROOT / 'checkpoints').glob('*.pt')]))

all_cells = [
    "leakage",       # NB08
    "topology",      # NB07/NB08
    "temporal",      # scripts/retrain_best.py
    "A4 augment",    # NB09 Section 4
    "A5 normalise",  # NB09 Section 5
]
print("\nAll metrics_all.csv experiment groups:")
print(df_all["experiment"].value_counts().to_string())


=== Milestone 3 acceptance check ===
  ST-GCN best test acc : 0.7716
  BiLSTM test acc      : 0.5660
  1D-CNN test acc      : 0.8650
  ST-GCN > BiLSTM?     : YES ✓
  ST-GCN > 1D-CNN?     : NO  (ok — see thesis rationale)

  Roadmap requirement: 'ST-GCN beats LSTM on ≥1 protocol' —  SATISFIED

Checkpoints in: ['adaptive_T32_full_bonelength.pt', 'adaptive_T32_full_raw.pt', 'adaptive_T32_full_torso.pt', 'adaptive_T32_none_torso.pt', 'adaptive_T32_spatial_torso.pt', 'adaptive_T32_temporal_torso.pt', 'hands42_T32_full_torso.pt', 'latefusion_T32_full_torso.pt']

All metrics_all.csv experiment groups:
experiment
ablation    8
topology    5
temporal    4
leakage     3
baseline    2


---
## Section 7 — Missing Checkpoints

The topology (single/dual at T=32) and temporal (T=48/64/96) experiments were run in NB07/NB08
but checkpoints were not saved. Re-run here via `run_training()` so every experiment has a `.pt` file
in `checkpoints/`. Results will match what is already in `metrics_all.csv` (same SEED=42 and
hyperparameters).

In [17]:
# ── Re-train topology experiments to save checkpoints ─────────────────────────
# single and dual at T=32 — results should match NB08 (same SEED, same budget)

for topo in ("single", "dual"):
    ckpt = PROJECT_ROOT / "checkpoints" / f"{topo}_T32_full_torso.pt"
    if ckpt.exists():
        print(f"{topo} checkpoint already exists — skip")
        continue
    print(f"\n{'='*55}")
    print(f"Re-training {topo} at T=32 for checkpoint")
    print(f"{'='*55}")
    r = run_training(
        topology      = topo,
        T             = 32,
        augmentation  = "full",
        normalisation = "torso",
        epochs        = 80,
        patience      = 10,
        verbose       = True,
    )
    print(f"  val={r['val_acc']:.4f}  test={r['test_acc']:.4f}  F1={r['macro_f1']:.4f}")

print("\nTopology checkpoints done.")


Re-training single at T=32 for checkpoint

  single_T32_full_torso  |  params=2,063,173  |  device=mps
  ep   1/80  tr=0.0065  vl=0.0271  (0.1m)
  ep   2/80  tr=0.0244  vl=0.0361  (0.3m)
  ep   3/80  tr=0.0223  vl=0.0542  (0.4m)
  ep   4/80  tr=0.0479  vl=0.0753  (0.6m)
  ep   5/80  tr=0.0483  vl=0.0723  (0.7m)
  ep   6/80  tr=0.0613  vl=0.1235  (0.9m)
  ep   7/80  tr=0.0861  vl=0.1114  (1.0m)
  ep   8/80  tr=0.0946  vl=0.0994  (1.1m)
  ep   9/80  tr=0.1133  vl=0.1687  (1.3m)
  ep  10/80  tr=0.1401  vl=0.1867  (1.4m)
  ep  11/80  tr=0.1706  vl=0.2289  (1.6m)
  ep  12/80  tr=0.1702  vl=0.2078  (1.7m)
  ep  13/80  tr=0.2076  vl=0.2108  (1.9m)
  ep  14/80  tr=0.2051  vl=0.2108  (2.0m)
  ep  15/80  tr=0.2319  vl=0.2590  (2.1m)
  ep  16/80  tr=0.2522  vl=0.2771  (2.3m)
  ep  17/80  tr=0.2498  vl=0.3163  (2.4m)
  ep  18/80  tr=0.2717  vl=0.2681  (2.5m)
  ep  19/80  tr=0.2868  vl=0.3102  (2.7m)
  ep  20/80  tr=0.3058  vl=0.3614  (2.8m)
  ep  21/80  tr=0.3123  vl=0.3434  (3.0m)
  ep  22/80  t

In [18]:
# ── Re-train temporal experiments to save checkpoints ─────────────────────────
# T=48, 64, 96 — adaptive topology

for T in (48, 64, 96):
    ckpt = PROJECT_ROOT / "checkpoints" / f"adaptive_T{T}_full_torso.pt"
    if ckpt.exists():
        print(f"T={T} checkpoint already exists — skip")
        continue
    print(f"\n{'='*55}")
    print(f"Re-training adaptive T={T} for checkpoint")
    print(f"{'='*55}")
    r = run_training(
        topology      = "adaptive",
        T             = T,
        augmentation  = "full",
        normalisation = "torso",
        epochs        = 80,
        patience      = 10,
        verbose       = True,
    )
    print(f"  val={r['val_acc']:.4f}  test={r['test_acc']:.4f}  F1={r['macro_f1']:.4f}")

print("\nTemporal checkpoints done.")
print("\nAll checkpoints in checkpoints/:")
for p in sorted((PROJECT_ROOT / "checkpoints").glob("*.pt")):
    print(f"  {p.name}")


Re-training adaptive T=48 for checkpoint

  adaptive_T48_full_torso  |  params=2,495,370  |  device=mps
  ep   1/80  tr=0.0045  vl=0.0211  (0.3m)
  ep   2/80  tr=0.0122  vl=0.0151  (0.6m)
  ep   3/80  tr=0.0158  vl=0.0181  (0.9m)
  ep   4/80  tr=0.0280  vl=0.0512  (1.2m)
  ep   5/80  tr=0.0284  vl=0.0241  (1.5m)
  ep   6/80  tr=0.0406  vl=0.0753  (1.8m)
  ep   7/80  tr=0.0556  vl=0.0753  (2.1m)
  ep   8/80  tr=0.0650  vl=0.0633  (2.4m)
  ep   9/80  tr=0.0800  vl=0.1265  (2.7m)
  ep  10/80  tr=0.1048  vl=0.1114  (3.0m)
  ep  11/80  tr=0.1223  vl=0.1355  (3.3m)
  ep  12/80  tr=0.1466  vl=0.1476  (3.6m)
  ep  13/80  tr=0.1734  vl=0.1687  (3.9m)
  ep  14/80  tr=0.2104  vl=0.1777  (4.2m)
  ep  15/80  tr=0.2311  vl=0.2018  (4.5m)
  ep  16/80  tr=0.2620  vl=0.2741  (4.8m)
  ep  17/80  tr=0.2766  vl=0.3283  (5.1m)
  ep  18/80  tr=0.3221  vl=0.3193  (5.4m)
  ep  19/80  tr=0.3193  vl=0.3765  (5.7m)
  ep  20/80  tr=0.3501  vl=0.2560  (6.0m)
  ep  21/80  tr=0.3798  vl=0.3072  (6.3m)
  ep  22/80  

---
## Section 8 — Multi-Seed Reproducibility (3 Seeds)

The roadmap requires `metrics_all.csv` to report results across **3 seeds** per experiment
(SEED = 42, 0, 1). Run the two most important experiments — the best model (adaptive T=32 full aug)
and the no-augmentation baseline — under seeds 0 and 1 (seed 42 already done).

Report: mean ± std across seeds. This confirms results are not seed-lucky.

In [19]:
import numpy as np

# Experiments to replicate across seeds
_MULTI_SEED_RUNS = [
    dict(topology="adaptive", T=32, augmentation="full",  normalisation="torso"),
    dict(topology="adaptive", T=32, augmentation="none",  normalisation="torso"),
]
SEEDS = [42, 0, 1]

seed_results = {cfg["augmentation"]: [] for cfg in _MULTI_SEED_RUNS}

for cfg in _MULTI_SEED_RUNS:
    aug = cfg["augmentation"]
    print(f"\n{'='*60}")
    print(f"Multi-seed: adaptive T=32 aug={aug}")
    print(f"{'='*60}")
    for seed in SEEDS:
        tag = f"seed{seed}"
        ckpt = PROJECT_ROOT / "checkpoints" / f"adaptive_T32_{aug}_torso_{tag}.pt"
        if ckpt.exists():
            import torch
            saved = torch.load(ckpt, map_location="cpu")
            r = dict(val_acc=saved["val_acc"], test_acc=saved["test_acc"], macro_f1=saved["macro_f1"])
            print(f"  seed={seed} — loaded from cache: test={r['test_acc']:.4f}")
        else:
            r = run_training(seed=seed, verbose=False, **cfg)
            # Save with seed-tagged name
            import torch, shutil
            generic_ckpt = PROJECT_ROOT / "checkpoints" / f"adaptive_T32_{aug}_torso.pt"
            shutil.copy(generic_ckpt, ckpt)
            print(f"  seed={seed}: val={r['val_acc']:.4f}  test={r['test_acc']:.4f}  F1={r['macro_f1']:.4f}")
        seed_results[aug].append(r)

# Report mean ± std
print("\n=== Multi-Seed Summary ===")
for aug, runs in seed_results.items():
    test_accs = [r["test_acc"] for r in runs]
    f1s       = [r["macro_f1"] for r in runs]
    print(f"\n  adaptive T=32 aug={aug}:")
    for seed, r in zip(SEEDS, runs):
        print(f"    seed={seed}: test={r['test_acc']:.4f}  F1={r['macro_f1']:.4f}")
    print(f"    → test_acc: {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
    print(f"    → macro_f1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")


Multi-seed: adaptive T=32 aug=full
  seed=42: val=0.7259  test=0.7179  F1=0.6996
  seed=0: val=0.7410  test=0.7261  F1=0.7243
  seed=1: val=0.7259  test=0.7040  F1=0.6946

Multi-seed: adaptive T=32 aug=none
  seed=42: val=0.9127  test=0.8928  F1=0.8925
  seed=0: val=0.8886  test=0.9056  F1=0.8962
  seed=1: val=0.9096  test=0.8986  F1=0.8894

=== Multi-Seed Summary ===

  adaptive T=32 aug=full:
    seed=42: test=0.7179  F1=0.6996
    seed=0: test=0.7261  F1=0.7243
    seed=1: test=0.7040  F1=0.6946
    → test_acc: 0.7160 ± 0.0091
    → macro_f1: 0.7062 ± 0.0130

  adaptive T=32 aug=none:
    seed=42: test=0.8928  F1=0.8925
    seed=0: test=0.9056  F1=0.8962
    seed=1: test=0.8986  F1=0.8894
    → test_acc: 0.8990 ± 0.0052
    → macro_f1: 0.8927 ± 0.0028


---
## Section 9 — A2: Single vs. Two-Hand Subset Analysis

The roadmap requires A2 to **report accuracy on single-hand vs. two-hand sign subsets**.
We approximate hand usage from the mean absolute value of left/right hand joints in the test set:
if a class has near-zero left-hand signal → right-hand dominant (single), and vice versa.
Classes where both hands are active → two-handed.

In [44]:
import numpy as np, torch, sys
from pathlib import Path
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from train import PROC_DIR, N_CLASSES, get_device, set_seed, build_model
from evaluate import load_test_arrays, run_inference

# ── Classify each class as single-hand or two-hand (velocity-based) ──────────
X_te = np.load(PROC_DIR / "X_sd_test.npy")   # (N, T, 53, 3)
y_te = np.load(PROC_DIR / "y_sd_test.npy")

# Frame-to-frame motion magnitude per hand per sample
l_hand_vel = np.abs(np.diff(X_te[:, :, :21,  :], axis=1)).mean(axis=(1, 2, 3))   # (N,)
r_hand_vel = np.abs(np.diff(X_te[:, :, 21:42,:], axis=1)).mean(axis=(1, 2, 3))   # (N,)

hand_type = {}   # class_idx → "two_hand" | "single_hand"
for cls in range(N_CLASSES):
    mask = y_te == cls
    if mask.sum() == 0:
        continue
    l_mean = l_hand_vel[mask].mean()
    r_mean = r_hand_vel[mask].mean()
    ratio  = min(l_mean, r_mean) / (max(l_mean, r_mean) + 1e-9)
    # Idle hand velocity < 50% of active hand → single-hand class
    hand_type[cls] = "two_hand" if ratio > 0.50 else "single_hand"

two_hand_classes    = {c for c, t in hand_type.items() if t == "two_hand"}
single_hand_classes = {c for c, t in hand_type.items() if t == "single_hand"}
print(f"Two-hand classes : {len(two_hand_classes)}")
print(f"Single-hand classes: {len(single_hand_classes)}")

single_mask = np.isin(y_te, list(single_hand_classes))
two_mask    = np.isin(y_te, list(two_hand_classes))
print(f"Test samples — single: {single_mask.sum()}, two: {two_mask.sum()}")

# ── Evaluate all topology checkpoints on each subset ─────────────────────────
topology_ckpts = {
    "single (53j fixed-uniform)": "single_T32_full_torso.pt",
    "dual (53j fixed-spatial)":   "dual_T32_full_torso.pt",
    "adaptive (53j learnable)":   "adaptive_T32_full_torso.pt",
    "hands42":                    "hands42_T32_full_torso.pt",
}

device = get_device()
set_seed(42)

a2_subset_rows = []
for label, ckpt_name in topology_ckpts.items():
    ckpt_path = PROJECT_ROOT / "checkpoints" / ckpt_name
    if not ckpt_path.exists():
        print(f"  SKIP {ckpt_name} — not found (run Section 7 first)")
        continue
    ckpt  = torch.load(ckpt_path, map_location="cpu")
    model = build_model(ckpt["topology"], device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    X_sub = X_te[:, :, :42, :] if ckpt["topology"] in ("hands42", "latefusion") else X_te
    y_pred, _, _ = run_inference(model, X_sub, ckpt["T"], device)

    overall = (y_pred == y_te).mean()
    s_acc = (y_pred[single_mask] == y_te[single_mask]).mean() if single_mask.sum() > 0 else float("nan")
    t_acc = (y_pred[two_mask]    == y_te[two_mask]).mean()    if two_mask.sum()    > 0 else float("nan")

    a2_subset_rows.append(dict(
        topology=label,
        overall=round(overall, 4),
        single_hand=round(s_acc, 4),
        two_hand=round(t_acc, 4),
        single_n=int(single_mask.sum()),
        two_n=int(two_mask.sum()),
    ))
    print(f"  {label}: overall={overall:.4f}  single={s_acc:.4f}  two={t_acc:.4f}")

a2_sub_df = pd.DataFrame(a2_subset_rows)
print("\n=== A2 — Accuracy by Hand-Usage Subset ===")
print(a2_sub_df.to_string(index=False))
a2_sub_df.to_csv(PROJECT_ROOT / "results" / "ablation_A2_topology_subset.csv", index=False)
print("\nSaved results/ablation_A2_topology_subset.csv")

Two-hand classes : 240
Single-hand classes: 20
Test samples — single: 44, two: 814
  single (53j fixed-uniform): overall=0.6655  single=0.5909  two=0.6695
  dual (53j fixed-spatial): overall=0.6585  single=0.6818  two=0.6572
  adaptive (53j learnable): overall=0.7063  single=0.7500  two=0.7039
  hands42: overall=0.7121  single=0.7273  two=0.7113

=== A2 — Accuracy by Hand-Usage Subset ===
                  topology  overall  single_hand  two_hand  single_n  two_n
single (53j fixed-uniform)   0.6655       0.5909    0.6695        44    814
  dual (53j fixed-spatial)   0.6585       0.6818    0.6572        44    814
  adaptive (53j learnable)   0.7063       0.7500    0.7039        44    814
                   hands42   0.7121       0.7273    0.7113        44    814

Saved results/ablation_A2_topology_subset.csv


---
## Section 10 — Top-5 Accuracy for All Checkpoints

The roadmap requires Top-1 **and Top-5** accuracy for all models. Run evaluate.py on every checkpoint and collect both metrics.

In [45]:
from evaluate import evaluate as eval_ckpt

top5_rows = []
for ckpt_path in sorted((PROJECT_ROOT / "checkpoints").glob("*.pt")):
    if "seed" in ckpt_path.name:
        continue   # skip per-seed duplicates
    print(f"\n── {ckpt_path.name}")
    result = eval_ckpt(ckpt_path, update_csv=False)
    top5_rows.append(dict(
        checkpoint=ckpt_path.name,
        top1_acc=round(result["test_acc"], 4),
        top5_acc=round(result["top5_acc"], 4),
        macro_f1=round(result["macro_f1"], 4),
    ))

top5_df = pd.DataFrame(top5_rows).sort_values("top1_acc", ascending=False)
print("\n=== Top-1 and Top-5 Accuracy — All Checkpoints ===")
print(top5_df.to_string(index=False))
top5_df.to_csv(PROJECT_ROOT / "results" / "top5_accuracy_all.csv", index=False)
print("\nSaved results/top5_accuracy_all.csv")


── adaptive_T32_full_bonelength.pt

Evaluating: adaptive_T32_full_bonelength.pt
  topology=adaptive  T=32  normalisation=bonelength

  Test Top-1 acc: 0.4942
  Test Top-5 acc: 0.7786
  Macro-F1      : 0.4709
  (stored val)  : 0.4608433734939759

  10 worst classes (by F1):
 idx          name  f1  support
 163    60. Mother 0.0      4.0
  38    20. female 0.0      2.0
  80      33. dead 0.0      1.0
 125     48. Green 0.0      4.0
 122       47. Red 0.0      5.0
 121      47. Gift 0.0      1.0
 119  46. Clothing 0.0      4.0
 188    71. Friday 0.0      1.0
 220 83. Afternoon 0.0      1.0
  51     25. Chair 0.0      1.0

  10 best classes (by F1):
 idx         name  f1  support
  99       4. sad 1.0      1.0
  36    20. Price 1.0      1.0
  31    19. House 1.0      4.0
  29   18. curved 1.0      1.0
 187 70. Thursday 1.0      1.0
  26     17. flat 1.0      1.0
 236   89. Waiter 1.0      2.0
 238      9. Nice 1.0      1.0
 183      7. Deaf 1.0      1.0
 161     6. Mouse 1.0      4.0

  C